# Inspect a trained GUFU autoencoder

This notebook finds the newest valid, trained AE checkpoint under the configured search roots, prepares one raw input batch, projects it into that checkpoint's exact BSSID vocabulary, and runs inference without training or writing artifacts. Input coordinates are retained only as metadata and are never passed into the model.

In [1]:
# import glob
# import pickle

# with open("/home/oem/GUFU/output.bkp/GF/raw_data/_observations_GF.pkl", "rb") as f:
#     tt = pickle.load(f)

In [2]:
from pathlib import Path
import torch
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "ae.py").is_file():
    raise RuntimeError("Run this notebook with the GUFU repository as the working directory.")

# Edit these paths to inspect another raw batch or another checkpoint directory.
CHECKPOINT_SEARCH_ROOTS = [REPO_ROOT / "output"]
INPUT_BATCH_PATHS = [REPO_ROOT / "input" / "GF" / "fingerprint0.txt"]
INPUT_FORMAT = "gufu_db"  # "gufu_db" or "gufu_jsonl"
INPUT_BATCH_ID = "inspect-fingerprint"
BATCH_START = 0
BATCH_SIZE = 4
SCAN_INDEX_IN_BATCH = 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

for path in INPUT_BATCH_PATHS:
    if not path.is_file():
        raise FileNotFoundError(f"Input batch file does not exist: {path}")
print(f"Device: {DEVICE}")
print(f"Input files: {INPUT_BATCH_PATHS}")


Device: cuda
Input files: [PosixPath('/home/oem/GUFU/input/GF/fingerprint0.txt')]


## Load the newest trained AE checkpoint

Candidates are checked with the AE loader; unrelated or invalid .pt files are skipped. The selected checkpoint must contain at least one completed training epoch.

In [3]:
from ae import load_ae_checkpoint

candidate_paths = sorted(
    (
        path
        for root in CHECKPOINT_SEARCH_ROOTS
        if root.is_dir()
        for path in root.rglob("*.pt")
        if "ae" in path.name.lower()
        and not any(part.lower().startswith(".staging") for part in path.parts)
    ),
    key=lambda path: path.stat().st_mtime_ns,
    reverse=True,
)

ae_checkpoint = None
skipped_checkpoints = []
for candidate in candidate_paths:
    try:
        loaded = load_ae_checkpoint(candidate, device=DEVICE)
    except Exception as exc:
        skipped_checkpoints.append((candidate, f"{type(exc).__name__}: {exc}"))
        continue
    if loaded.manifest.epochs_completed < 1:
        skipped_checkpoints.append((candidate, "checkpoint has no completed training epoch"))
        continue
    ae_checkpoint = loaded
    break

if ae_checkpoint is None:
    details = "\n".join(f"  {path}: {reason}" for path, reason in skipped_checkpoints)
    raise FileNotFoundError(
        "No trained, loadable AE checkpoint was found under "
        f"{CHECKPOINT_SEARCH_ROOTS}.\n{details}"
    )

AE_CHECKPOINT_PATH = ae_checkpoint.path
checkpoint_info = pd.Series(
    {
        "checkpoint": str(AE_CHECKPOINT_PATH),
        "modified": pd.Timestamp(AE_CHECKPOINT_PATH.stat().st_mtime, unit="s"),
        "sha256": ae_checkpoint.checkpoint_sha256,
        "dataset_id": ae_checkpoint.manifest.training_dataset.dataset_id,
        "training_batch_id": ae_checkpoint.manifest.training_dataset.batch_id,
        "epochs_completed": ae_checkpoint.manifest.epochs_completed,
        "training_loss": ae_checkpoint.manifest.training_config.loss,
        "loss_history_definition": ae_checkpoint.manifest.loss_history_definition,
        "normalization": ae_checkpoint.manifest.normalization,
        "input_AP_count": ae_checkpoint.manifest.model.input_dim,
        "latent_dimension": ae_checkpoint.manifest.model.latent_dim,
        "device": DEVICE,
    },
    name="selected_AE_checkpoint",
)
display(checkpoint_info.to_frame())
if skipped_checkpoints:
    print(f"Skipped {len(skipped_checkpoints)} invalid or untrained checkpoint candidate(s).")


,selected_AE_checkpoint
checkpoint,/home/oem/GUFU/output/notebook_runs/GF/real-GF...
modified,2026-10-13 08:30:07.112124205
sha256,b7a11bc1558e65dd3f64cedd411497f645794810264b6a...
dataset_id,GF
training_batch_id,fingerprint0
epochs_completed,500
training_loss,mean_squared_error
loss_history_definition,element_weighted_batch_mse_per_epoch
normalization,paper_ae
input_AP_count,673


## Prepare and align the input batch

The raw batch may contain APs absent from the trained vocabulary. We preserve that full prepared batch, then make an explicit shared-vocabulary projection for AE inference. The AE never receives a resized input or columns in a different order.

In [4]:
from batch_transition import project_prepared_batch_to_vocabulary
from gufu_data import PreprocessingConfig, prepare_dataset_files

full_input_batch = prepare_dataset_files(
    INPUT_BATCH_PATHS,
    PreprocessingConfig(
        dataset_id=ae_checkpoint.manifest.training_dataset.dataset_id,
        batch_id=INPUT_BATCH_ID,
        batch_role="update_batch",
        input_format=INPUT_FORMAT,
        base_bssid_vocabulary=ae_checkpoint.manifest.bssid_vocabulary,
        normalization_mode=ae_checkpoint.manifest.normalization,
    ),
)
ae_input_batch = project_prepared_batch_to_vocabulary(
    full_input_batch,
    ae_checkpoint.manifest.bssid_vocabulary,
)

# Re-load with the prepared input to enforce dataset, vocabulary, normalization,
# and model-width compatibility before forwarding any data.
ae_checkpoint = load_ae_checkpoint(
    AE_CHECKPOINT_PATH, ae_input_batch, device=DEVICE
)
expected_vocab = ae_checkpoint.manifest.bssid_vocabulary
assert ae_input_batch.config.dataset_id == ae_checkpoint.manifest.training_dataset.dataset_id
assert ae_input_batch.config.normalization_mode == ae_checkpoint.manifest.normalization
assert ae_input_batch.bssid_vocabulary == expected_vocab
assert len(expected_vocab) == ae_checkpoint.manifest.model.input_dim

observed_in_batch = {
    bssid
    for scan in full_input_batch.scans
    for bssid, observed in zip(full_input_batch.bssid_vocabulary, scan.observation_mask)
    if observed
}
shared_bssids = tuple(sorted(observed_in_batch & set(expected_vocab)))
new_bssids = tuple(sorted(observed_in_batch - set(expected_vocab)))

print(f"Full prepared input: {full_input_batch.summary()}")
print(f"AE projection: {ae_input_batch.summary()}")
print(f"Shared APs observed in this batch: {len(shared_bssids)}")
print(f"New APs retained for other consumers, excluded from AE input: {len(new_bssids)}")
print(f"Projected vocabulary matches checkpoint: {ae_input_batch.bssid_vocabulary == expected_vocab}")


Full prepared input: {'dataset_id': 'GF', 'batch_id': 'inspect-fingerprint', 'batch_role': 'update_batch', 'scan_count': 392, 'ap_count': 673, 'reference_label_count': 0, 'evaluation_truth_count': 392, 'dropped_record_count': 0}
AE projection: {'dataset_id': 'GF', 'batch_id': 'inspect-fingerprint', 'batch_role': 'update_batch', 'scan_count': 392, 'ap_count': 673, 'reference_label_count': 0, 'evaluation_truth_count': 392, 'dropped_record_count': 0}
Shared APs observed in this batch: 673
New APs retained for other consumers, excluded from AE input: 0
Projected vocabulary matches checkpoint: True


## Run one minibatch through the encoder and decoder

The exposed tensors remain in AP vocabulary order. Variables are left in the notebook namespace for interactive inspection: input values, latent vectors, reconstructed values, and residuals.

In [5]:
from ae import ae_input_values
from gufu_data import ae_value_to_raw_dbm

model = ae_checkpoint.model
model.eval()
all_input_values = torch.tensor(
    ae_input_values(ae_input_batch),
    dtype=next(model.parameters()).dtype,
    device=DEVICE,
)
if all_input_values.ndim != 2 or all_input_values.shape != (
    len(ae_input_batch.scans),
    len(expected_vocab),
):
    raise AssertionError("prepared AE matrix does not match scan and vocabulary dimensions")

batch_end = min(BATCH_START + BATCH_SIZE, len(ae_input_batch.scans))
if BATCH_START < 0 or BATCH_START >= batch_end:
    raise ValueError("BATCH_START must select at least one retained scan")
input_batch_tensor = all_input_values[BATCH_START:batch_end].unsqueeze(1)
if input_batch_tensor.shape[1:] != (1, ae_checkpoint.manifest.model.input_dim):
    raise AssertionError(f"unexpected model input shape: {tuple(input_batch_tensor.shape)}")

with torch.inference_mode():
    latent_batch, reconstruction_batch = model(input_batch_tensor)
residual_batch = reconstruction_batch - input_batch_tensor

expected_rows = batch_end - BATCH_START
assert latent_batch.shape == (expected_rows, 1, ae_checkpoint.manifest.model.latent_dim)
assert reconstruction_batch.shape == input_batch_tensor.shape
assert torch.isfinite(latent_batch).all()
assert torch.isfinite(reconstruction_batch).all()

input_masks = torch.tensor(
    [scan.observation_mask for scan in ae_input_batch.scans[BATCH_START:batch_end]],
    dtype=torch.bool,
    device=DEVICE,
)
frobenius_by_scan = torch.linalg.vector_norm(residual_batch, dim=(1, 2))
rmse_all_by_scan = torch.sqrt(torch.mean(residual_batch.square(), dim=(1, 2)))
rmse_observed_by_scan = []
for row in range(expected_rows):
    observed_residuals = residual_batch[row, 0, input_masks[row]]
    rmse_observed_by_scan.append(
        float(torch.sqrt(torch.mean(observed_residuals.square())).cpu())
        if observed_residuals.numel()
        else float("nan")
    )

print(f"input_batch_tensor: {tuple(input_batch_tensor.shape)}")
print(f"latent_batch: {tuple(latent_batch.shape)}")
print(f"reconstruction_batch: {tuple(reconstruction_batch.shape)}")
print("All-entry RMSE includes missing AP slots filled using the checkpoint's normalization.")


input_batch_tensor: (4, 1, 673)
latent_batch: (4, 1, 64)
reconstruction_batch: (4, 1, 673)
All-entry RMSE includes missing AP slots filled using the checkpoint's normalization.


In [ ]:
from copy import deepcopy

output = deepcopy(reconstruction_batch)
output[output<1e-3] = 0.0

print(output[0])

tensor([[0.0000, 0.0000, 0.0159, 0.0100, 0.0114, 0.0000, 0.0011, 0.0051, 0.0000,
         0.0000, 0.0034, 0.0000, 0.0071, 0.0000, 0.0041, 0.0153, 0.0000, 0.1965,
         0.0000, 0.0000, 0.0000, 0.0082, 0.0000, 0.0000, 0.0000, 0.0000, 0.0507,
         0.0412, 0.0000, 0.0088, 0.0043, 0.0194, 0.0103, 0.0219, 0.0000, 0.0000,
         0.0456, 0.0000, 0.0057, 0.0055, 0.0043, 0.0000, 0.0043, 0.0000, 0.0026,
         0.0278, 0.0000, 0.0158, 0.0213, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0108, 0.0088, 0.2975, 0.0000, 0.0057, 0.0089, 0.2326, 0.0000,
         0.0000, 0.0000, 0.0123, 0.0060, 0.0000, 0.0287, 0.0202, 0.0000, 0.0123,
         0.0050, 0.0108, 0.0000, 0.0000, 0.1355, 0.0501, 0.0357, 0.0000, 0.0000,
         0.0000, 0.0207, 0.0260, 0.0057, 0.0000, 0.0708, 0.0000, 0.0025, 0.0217,
         0.0083, 0.0138, 0.0000, 0.0000, 0.0029, 0.0780, 0.0000, 0.0000, 0.0083,
         0.0215, 0.0000, 0.0037, 0.0000, 0.0000, 0.0126, 0.0269, 0.0000, 0.0000,
         0.1191, 0.0422, 0.1

In [8]:
print(input_batch_tensor[0])

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.3673,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.3340, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.3417, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.3417, 0.0000, 0.0

In [7]:
blah

NameError: name 'blah' is not defined

## Inspect per-scan metrics, latent values, and AP-level reconstruction

Choose any row in the forwarded minibatch. The tables show the full raw input AP set, which APs entered the fixed-width model, and the reconstructed values for the checkpoint vocabulary. Missing measurements are marked NaN in measured-only columns; the explicit mask is authoritative.

In [ ]:
batch_scan_rows = []
for local_row, scan in enumerate(ae_input_batch.scans[BATCH_START:batch_end]):
    batch_scan_rows.append(
        {
            "batch_row": local_row,
            "scan_id": scan.scan_id,
            "source_file": scan.source.source_file,
            "source_row": scan.source.source_row,
            "coordinate_role": scan.coordinate_role,
            "has_coordinates": scan.has_coordinates,
            "observed_AP_count": sum(scan.observation_mask),
            "frobenius_norm": float(frobenius_by_scan[local_row].cpu()),
            "rmse_all_AP_slots": float(rmse_all_by_scan[local_row].cpu()),
            "rmse_observed_APs": rmse_observed_by_scan[local_row],
        }
    )
batch_scan_summary = pd.DataFrame(batch_scan_rows)
display(batch_scan_summary)

if not 0 <= SCAN_INDEX_IN_BATCH < expected_rows:
    raise IndexError(f"SCAN_INDEX_IN_BATCH must be between 0 and {expected_rows - 1}")
selected_local_row = SCAN_INDEX_IN_BATCH
selected_scan = ae_input_batch.scans[BATCH_START + selected_local_row]

input_row = input_batch_tensor[selected_local_row, 0].detach().cpu().tolist()
reconstruction_row = reconstruction_batch[selected_local_row, 0].detach().cpu().tolist()
observation_mask_row = selected_scan.observation_mask

ap_inspection = pd.DataFrame(
    {
        "ap_id": ae_input_batch.ap_ids,
        "bssid": ae_input_batch.bssid_vocabulary,
        "observed": observation_mask_row,
        "measured_rssi_dbm": [
            rssi if observed else float("nan")
            for rssi, observed in zip(selected_scan.rss_dbm, observation_mask_row)
        ],
        "ae_input": input_row,
        "reconstructed_ae": reconstruction_row,
        "reconstructed_rssi_dbm": [
            ae_value_to_raw_dbm(value, ae_checkpoint.manifest.normalization)
            for value in reconstruction_row
        ],
        "reconstruction_error_db_measured_only": [
            (
                ae_value_to_raw_dbm(predicted, ae_checkpoint.manifest.normalization) - measured
                if observed
                else float("nan")
            )
            for predicted, measured, observed in zip(
                reconstruction_row, selected_scan.rss_dbm, observation_mask_row
            )
        ],
    }
)
latent_inspection = pd.DataFrame(
    {
        "latent_index": range(latent_batch.shape[-1]),
        "value": latent_batch[selected_local_row, 0].detach().cpu().tolist(),
    }
)

full_selected_scan = full_input_batch.scans[BATCH_START + selected_local_row]
ae_column_by_bssid = {bssid: column for column, bssid in enumerate(expected_vocab)}
full_input_ap_inspection = pd.DataFrame(
    {
        "ap_id": full_input_batch.ap_ids,
        "bssid": full_input_batch.bssid_vocabulary,
        "observed": full_selected_scan.observation_mask,
        "raw_rssi_dbm": [
            rssi if observed else float("nan")
            for rssi, observed in zip(
                full_selected_scan.rss_dbm, full_selected_scan.observation_mask
            )
        ],
        "AE_vocabulary_column": [
            ae_column_by_bssid.get(bssid, float("nan"))
            for bssid in full_input_batch.bssid_vocabulary
        ],
        "passed_to_AE": [
            bssid in ae_column_by_bssid
            for bssid in full_input_batch.bssid_vocabulary
        ],
    }
)

print(f"Detailed scan: {selected_scan.scan_id}")
print("Latent coordinates:")
display(latent_inspection)
print("Full raw input AP set, including APs excluded from this fixed-width AE:")
display(full_input_ap_inspection)
print("AE vocabulary input and reconstruction:")
display(ap_inspection)


,batch_row,scan_id,source_file,source_row,coordinate_role,has_coordinates,observed_AP_count,frobenius_norm,rmse_all_AP_slots,rmse_observed_APs
0,0,scan:e4dad4744e8abc2e3b590a909e59f60f071dbade6...,fingerprint0.txt,1,evaluation_only,True,54,0.916412,0.035325,0.101703
1,1,scan:a03debae8e9e5112e387ef7d17d19a95adf5f1fa4...,fingerprint0.txt,2,evaluation_only,True,46,0.917759,0.035377,0.077055
2,2,scan:5f680352946fbc97d8cff0b936657814d0a1bf84b...,fingerprint0.txt,3,evaluation_only,True,62,2.152750,0.082982,0.222251
3,3,scan:fb36c03cbbfaf4e57ca2377b6e66dfeecfebbb859...,fingerprint0.txt,4,evaluation_only,True,69,2.248547,0.086675,0.236641


Detailed scan: scan:e4dad4744e8abc2e3b590a909e59f60f071dbade62a85018650bdfa6332a5e15
Latent coordinates:


,latent_index,value
0,0,-0.244716
1,1,-0.244716
2,2,-0.244716
3,3,-0.244716
4,4,-0.244716
5,5,-0.244716
6,6,-0.244716
7,7,-0.244716
8,8,-0.244716
9,9,-0.244716


Full raw input AP set, including APs excluded from this fixed-width AE:


,ap_id,bssid,observed,raw_rssi_dbm,AE_vocabulary_column,passed_to_AE
0,ap:000B819EEBA5,000B819EEBA5,False,NaN,0,True
1,ap:000B819FDD48,000B819FDD48,False,NaN,1,True
2,ap:000B8575C2CF,000B8575C2CF,False,NaN,2,True
3,ap:000B8575E238,000B8575E238,False,NaN,3,True
4,ap:000B8575E23F,000B8575E23F,False,NaN,4,True
...,...,...,...,...,...,...
668,ap:F8E903BC0A02,F8E903BC0A02,False,NaN,668,True
669,ap:FADA0C542CCD,FADA0C542CCD,False,NaN,669,True
670,ap:FCECDA17AA68,FCECDA17AA68,False,NaN,670,True
671,ap:FCECDA8AEA18,FCECDA8AEA18,False,NaN,671,True


AE vocabulary input and reconstruction:


,ap_id,bssid,observed,measured_rssi_dbm,ae_input,reconstructed_ae,reconstructed_rssi_dbm,reconstruction_error_db_measured_only
0,ap:000B819EEBA5,000B819EEBA5,False,NaN,0.0,0.004781,-119.426294,NaN
1,ap:000B819FDD48,000B819FDD48,False,NaN,0.0,0.000926,-119.888856,NaN
2,ap:000B8575C2CF,000B8575C2CF,False,NaN,0.0,-0.006670,-120.800347,NaN
3,ap:000B8575E238,000B8575E238,False,NaN,0.0,-0.004841,-120.580907,NaN
4,ap:000B8575E23F,000B8575E23F,False,NaN,0.0,0.003088,-119.629459,NaN
...,...,...,...,...,...,...,...,...
668,ap:F8E903BC0A02,F8E903BC0A02,False,NaN,0.0,-0.002204,-120.264436,NaN
669,ap:FADA0C542CCD,FADA0C542CCD,False,NaN,0.0,0.003748,-119.550190,NaN
670,ap:FCECDA17AA68,FCECDA17AA68,False,NaN,0.0,-0.002589,-120.310687,NaN
671,ap:FCECDA8AEA18,FCECDA8AEA18,False,NaN,0.0,0.004420,-119.469600,NaN
